<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_39_middleware_redis/note_lesson_39_redis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ Урок 39 — Middleware і Redis: новинний агрегатор, крок 4

| Крок | Що робимо |
|---|---|
| 0 | `news_hub`: база SQLite (Alembic) і Redis у пам'яті (`fakeredis`) |
| 1 | middleware: заголовки й порядок шарів (вправи 1–2) |
| 2 | кеш стрічки: hit/miss, ключі, інвалідація (вправи 3–5) |
| 3 | rate limit: 429 і ключ без TTL (вправи 6–7) |
| 4 | фоновий збір: 202 і статус (вправа 8) |
| 5 | ковзне вікно (вправа 9) |
| 6 | знайди помилку: сесія запиту у фоні (вправа 10) |

**Як працювати:** зверху вниз; перед **🔮 Прогнозом** спершу відповідай сам. Рефакторинги, діаграми і вивід на справжніх PostgreSQL і Redis — у книзі курсу: [Урок 39](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_39/).

---
## 0. Проєкт, база і Redis

`news_hub` з уроку 38 + `cache.py` (Redis, кеш), `middleware.py` (id і час запиту, rate limit, інвалідація кешу), `jobs.py` (фоновий збір).

Змінні середовища — **до** імпорту `news_hub`: база — файл SQLite, Redis — `fakeredis://` (Redis у пам'яті процесу, сервер не потрібен).

In [ ]:
import os
import subprocess
import sys

LESSON = "module_4/lessons/lesson_39_middleware_redis"
REPO = "https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026.git"

if not os.path.isdir("news_hub"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))

try:
    import aiosqlite, alembic, fakeredis, fastapi, redis, sqlalchemy  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "news_hub/requirements.txt"], check=True)

os.chdir("news_hub")
sys.path.insert(0, os.getcwd())

os.environ["DATABASE_URL"] = "sqlite+aiosqlite:///lesson39.db"
os.environ["REDIS_URL"] = "fakeredis://"
if os.path.exists("lesson39.db"):
    os.remove("lesson39.db")
subprocess.run([sys.executable, "-m", "alembic", "upgrade", "head"], capture_output=True, check=True)

import asyncio
import time

import fakeredis
from fastapi import BackgroundTasks, FastAPI, Request
from fastapi.testclient import TestClient

from news_hub.api import JobStore, RedisDep, RepoDep, app
from news_hub.cache import NewsCache
from news_hub.db import SessionFactory
from news_hub.middleware import RATE_LIMIT_REQUESTS, RateLimiter
from news_hub.models import validate_news
from news_hub.repository import NewsRepository
from news_hub.snapshot import load_snapshot

client = TestClient(app)
client.__enter__()
redis = app.state.redis                           # fakeredis, створений у lifespan
print(client.get("/health").json(), "| ліміт:", RATE_LIMIT_REQUESTS, "POST /api/scrape* за хвилину")

---
## 1. Middleware

### Вправа 1

Надішли `GET /health` зі своїм заголовком `X-Request-ID: my-first-request` і збережи відповідь у `response`. Middleware `request_context` має повернути цей самий id і час обробки.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
response = client.get("/health", headers={"X-Request-ID": "my-first-request"})
# END SOLUTION

print(response.headers["X-Request-ID"], response.headers["X-Process-Time"])
assert response.headers["X-Request-ID"] == "my-first-request"
assert response.headers["X-Process-Time"].endswith("ms")
print("✅ Вправа 1 пройдена")

**🔮 Прогноз:** Два middleware: `outer` зареєстровано **останнім**, `inner` — першим. Кожен дописує в список `trace` рядок до `call_next` і після. Який вийде список?

<details>
<summary>Відповідь</summary>

`['outer до', 'inner до', 'ендпоінт', 'inner після', 'outer після']` — останній зареєстрований стає зовнішнім шаром «цибулі».

</details>

### Вправа 2

Перевір прогноз: у застосунку `onion` зареєструй `inner`, потім `outer` (через `onion.middleware("http")(…)`) і збережи в `trace` порядок виклику.

In [ ]:
onion = FastAPI()
trace = []


@onion.get("/")
async def root() -> dict[str, str]:
    trace.append("ендпоінт")
    return {"ok": "yes"}


async def inner(request: Request, call_next):
    trace.append("inner до")
    response = await call_next(request)
    trace.append("inner після")
    return response


async def outer(request: Request, call_next):
    trace.append("outer до")
    response = await call_next(request)
    trace.append("outer після")
    return response


# YOUR CODE HERE
# BEGIN SOLUTION
onion.middleware("http")(inner)
onion.middleware("http")(outer)
# END SOLUTION

TestClient(onion).get("/")
print(trace)
assert trace == ["outer до", "inner до", "ендпоінт", "inner після", "outer після"]
print("✅ Вправа 2 пройдена")

---
## 2. Кеш стрічки

`GET /api/news` і `/api/news/stats` кешуються в Redis на 60 с; заголовок `X-Cache` каже, звідки відповідь.

### Вправа 3

Збери знімок, потім **двічі** запитай `GET /api/news?lang=uk&limit=3`. Значення заголовка `X-Cache` обох відповідей — у список `states`.

In [ ]:
client.post("/api/scrape", json={"source": "snapshot"})

# YOUR CODE HERE
# BEGIN SOLUTION
states = [client.get("/api/news", params={"lang": "uk", "limit": 3}).headers["X-Cache"] for _ in range(2)]
# END SOLUTION

print(states)
assert states == ["MISS", "HIT"]
print("✅ Вправа 3 пройдена")

### Вправа 4

Ключ кешу не має залежати від **порядку** параметрів. Порахуй `NewsCache(redis).key(...)` для `{"lang": "uk", "limit": 3}` і `{"limit": 3, "lang": "uk"}` (корутини — через `client.portal.call`, бо Redis живе в циклі подій застосунку).

In [ ]:
cache = NewsCache(redis)

# YOUR CODE HERE
# BEGIN SOLUTION
key_a = client.portal.call(cache.key, "list", {"lang": "uk", "limit": 3})
key_b = client.portal.call(cache.key, "list", {"limit": 3, "lang": "uk"})
# END SOLUTION

print(key_a)
assert key_a == key_b and key_a.startswith("news:v")
print("✅ Вправа 4 пройдена")

**🔮 Прогноз:** Після `POST /api/news` (201) що покаже `X-Cache` для `/api/news/stats`, закешованого перед тим? А після невдалого `POST` (409)?

<details>
<summary>Відповідь</summary>

Після 201 — `MISS`: middleware `invalidate_cache` збільшив `news:version`, старий ключ більше не читається. Після 409 — `HIT`: запис не відбувся, версія не змінилась.

</details>

### Вправа 5

Перевір прогноз. Заголовки `X-Cache` для `/api/news/stats` після успішного й після невдалого створення — у `after_created` і `after_conflict`.

In [ ]:
NEW = {"title": "НБУ залишив облікову ставку без змін", "url": "https://www.rbc.ua/ukr/news/nbu-rate-778.html"}
client.get("/api/news/stats")

# YOUR CODE HERE
# BEGIN SOLUTION
client.post("/api/news", json=NEW)
after_created = client.get("/api/news/stats").headers["X-Cache"]
client.post("/api/news", json=NEW)
after_conflict = client.get("/api/news/stats").headers["X-Cache"]
# END SOLUTION

print(after_created, after_conflict, "| total =", client.get("/api/news/stats").json()["total"])
assert (after_created, after_conflict) == ("MISS", "HIT")
print("✅ Вправа 5 пройдена")

---
## 3. Rate limit

### Вправа 6

Ти вже зробив 1 `POST /api/scrape`. Надсилай ще, поки не отримаєш `429`; збережи статуси в `statuses`, а заголовок `Retry-After` відповіді з `429` — у `retry_after`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
statuses = []
while True:
    response = client.post("/api/scrape", json={"source": "snapshot"})
    statuses.append(response.status_code)
    if response.status_code == 429:
        retry_after = int(response.headers["Retry-After"])
        break
# END SOLUTION

print(statuses, "| Retry-After:", retry_after)
assert statuses == [200] * (RATE_LIMIT_REQUESTS - 1) + [429]
assert 0 < retry_after <= 60
print("✅ Вправа 6 пройдена")

### Вправа 7

Старий код (`ai_bot`) ставив TTL лише при `count == 1`: якщо процес падав між `INCR` і `EXPIRE`, ключ лишався без TTL — блокування назавжди. Створи такий «залишок» (`SET rate:scrape:crashed 99` без TTL), виклич `RateLimiter(redis).hit("crashed", "scrape")` і перевір, що ключ отримав TTL.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
client.portal.call(redis.set, "rate:scrape:crashed", 99)
ttl_before = client.portal.call(redis.ttl, "rate:scrape:crashed")
allowed, count, ttl_after = client.portal.call(RateLimiter(redis).hit, "crashed", "scrape")
# END SOLUTION

print("TTL до:", ttl_before, "| після:", ttl_after, "| count:", count, "| дозволено:", allowed)
assert ttl_before == -1 and 0 < ttl_after <= 60 and not allowed
print("✅ Вправа 7 пройдена")

---
## 4. Фоновий збір

`POST /api/scrape/jobs` → `202` і `job_id`; статус — `GET /api/scrape/jobs/{job_id}`. `TestClient` виконує фонові задачі до того, як повернути відповідь, тож тут статус уже фінальний (на справжньому сервері треба опитувати — сторінка уроку).

### Вправа 8

Ліміт з вправи 6 ще діє — скинь лічильник клієнта `testclient` (`DELETE rate:scrape:testclient`). Потім очисти новини, запусти фоновий збір знімка і збережи статус задачі в `job`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
client.portal.call(redis.delete, "rate:scrape:testclient")
client.delete("/api/news")
started = client.post("/api/scrape/jobs", json={"source": "snapshot"})
job = client.get(f"/api/scrape/jobs/{started.json()['job_id']}").json()
# END SOLUTION

print(started.status_code, {key: job[key] for key in ("status", "news_found", "news_saved")})
assert started.status_code == 202 and job["status"] == "done" and job["news_saved"] == 168
print("✅ Вправа 8 пройдена")

---
## 5. Ковзне вікно

Фіксоване вікно на межі пропускає до 2× ліміту. **Ковзне** рахує запити за останні `window` секунд на **sorted set**: бал — час запиту.

### Вправа 9

Допиши `sliding_hit`: в одній транзакції `ZREMRANGEBYSCORE` (прибрати старші за `now - window`), `ZADD` (поточний запит), `ZCARD` (скільки у вікні), `PEXPIRE`. Дозволено, якщо `ZCARD <= limit`.

In [ ]:
import uuid

sliding_redis = fakeredis.FakeAsyncRedis(decode_responses=True)


async def sliding_hit(key: str, limit: int, window_ms: int, now_ms: int) -> bool:
    # YOUR CODE HERE
    # BEGIN SOLUTION
    async with sliding_redis.pipeline(transaction=True) as pipe:
        pipe.zremrangebyscore(key, 0, now_ms - window_ms)
        pipe.zadd(key, {uuid.uuid4().hex: now_ms})
        pipe.zcard(key)
        pipe.pexpire(key, window_ms)
        _, _, count, _ = await pipe.execute()
    return count <= limit
    # END SOLUTION


# ліміт 3 за 2000 мс: запит на 0 мс, потім серія на «межі» 1900–2150 мс
times = [0, 1900, 1950, 2050, 2100, 2150]
results = [await sliding_hit("rate:sliding:demo", 3, 2000, t) for t in times]
print(list(zip(times, results)))
# 2050: запит з 0 мс уже поза вікном (50, 2050] → у вікні 1900, 1950, 2050 = 3 → дозволено;
# 2100 і 2150: у вікні 4 і 5 (відхилені теж рахуються) → ні. У будь-які 2 с — не більше 3 дозволених.
assert results == [True, True, True, True, False, False]
print("✅ Вправа 9 пройдена")

---
## 6. Знайди помилку

Фонова задача отримала **репозиторій запиту** (як старий `news_dashboard` передавав у задачу `db` запиту):

In [ ]:
async def scrape_in_background(repo: NewsRepository, jobs: JobStore, job_id: str) -> None:
    valid, _ = validate_news(load_snapshot())
    saved = await repo.add_many(valid)
    await jobs.update(job_id, status="done", news_saved=saved)


@app.post("/api/scrape/jobs-buggy", status_code=202)
async def start_buggy(background: BackgroundTasks, repo: RepoDep, redis: RedisDep) -> dict[str, str]:
    job = await JobStore(redis).create("snapshot", "async")
    background.add_task(scrape_in_background, repo, JobStore(redis), job.job_id)
    return {"job_id": job.job_id}


client.delete("/api/news")
job_id = client.post("/api/scrape/jobs-buggy").json()["job_id"]
print("задача:", client.get(f"/api/scrape/jobs/{job_id}").json()["news_saved"], "збережено")
print("у базі:", client.get("/api/news/count").json())

# «Воскресла» сесія досі тримає незакомічену транзакцію — у SQLite це блокування всієї бази на запис
# («database is locked» для наступного INSERT). Приберемо її збирачем сміття, щоб продовжити.
import gc
gc.collect()

### Вправа 10

Перепиши `scrape_in_background_fixed`: без репозиторію запиту — **своя** сесія (`SessionFactory()`), `NewsRepository(session).add_many(...)` і `commit()`. Зареєструй `/api/scrape/jobs-fixed` і перевір, що новини справді в базі.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
async def scrape_in_background_fixed(jobs: JobStore, job_id: str) -> None:
    valid, _ = validate_news(load_snapshot())
    async with SessionFactory() as session:
        saved = await NewsRepository(session).add_many(valid)
        await session.commit()
    await jobs.update(job_id, status="done", news_saved=saved)


@app.post("/api/scrape/jobs-fixed", status_code=202)
async def start_fixed(background: BackgroundTasks, redis: RedisDep) -> dict[str, str]:
    job = await JobStore(redis).create("snapshot", "async")
    background.add_task(scrape_in_background_fixed, JobStore(redis), job.job_id)
    return {"job_id": job.job_id}
# END SOLUTION

client.delete("/api/news")
job_id = client.post("/api/scrape/jobs-fixed").json()["job_id"]
saved = int(client.get(f"/api/scrape/jobs/{job_id}").json()["news_saved"])
count = client.get("/api/news/count").json()["count"]
print("задача:", saved, "| у базі:", count)
assert saved == count == 168
print("✅ Вправа 10 пройдена")

---
## Самоперевірка

1. У якому порядку виконуються middleware для запиту і для відповіді?
2. Чому версію кешу збільшують після COMMIT, а не в ендпоінті?
3. Чим небезпечні окремі `INCR` і `EXPIRE` у rate limit?
4. Що означає `202 Accepted` і як клієнт дізнається результат?
5. Чому фонова задача не може використати сесію бази з запиту?

<details>
<summary>Відповіді</summary>

1. Запит — від останнього зареєстрованого (зовнішнього) до першого; відповідь — навпаки.
2. COMMIT робить `get_db` після ендпоінта; версія до COMMIT дозволяє паралельному GET закешувати старі дані під новою версією.
3. Збій між ними — ключ без TTL, лічильник більше не обнуляється, клієнта заблоковано назавжди. Рішення — одна транзакція і `EXPIRE … NX`.
4. «Прийнято до роботи», а не «готово»; у відповіді `job_id`, клієнт опитує `GET /api/scrape/jobs/{job_id}`.
5. Задача виконується після відповіді, коли сесію вже закомічено й закрито; записи в ній ніхто не закомітить — дані тихо пропадуть.

</details>

## Далі

- Книга курсу: [Урок 39](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_39/) — «цибуля» middleware покроково, чому інвалідація після COMMIT, межа фіксованого вікна, polling на справжньому сервері.
- У папці `news_hub`: `docker compose up -d db redis`, `alembic upgrade head`, `uvicorn news_hub.api:app --reload`; `pytest`.
- **Урок 41** — тести API агрегатора.

In [ ]:
client.__exit__(None, None, None)